In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
import os
import json
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error
import cloudpickle
import io

# --- Include the functions from your Streamlit app for M1 ---
def calculate_current_month(row):
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    return (cohort_start + pd.DateOffset(months=int(elapsed_months))).month

def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)
        hess = np.ones_like(y_true)
        return grad, hess
    return custom_loss

def model_train_save_pipeline(game, alpha, X_train, y_train, X_test, y_test):
    model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True, random_state=42)
    numerical_columns = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), numerical_columns)],
        remainder='passthrough'
    )
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    param_dist = {
        'model__n_estimators': randint(50, 500),
        'model__learning_rate': uniform(0.01, 0.09),
        'model__max_depth': randint(3, 10),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__min_child_weight': randint(1, 10),
        'model__gamma': uniform(0, 0.3),
        'model__reg_alpha': uniform(0, 1),
        'model__reg_lambda': uniform(1, 2),
    }
    random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist,
                                         n_iter=100, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
                                         cv=TimeSeriesSplit(n_splits=5), n_jobs=-1, random_state=42)
    random_search.fit(X_train, y_train)
    best_model = random_search.best_estimator_
    with open(f'{game}_xgb_{alpha}.pkl', 'wb') as f:
        cloudpickle.dump(best_model, f)
    best_model.__dict__["feature_names_in_"] = X_train.columns
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))
    return {"model": best_model, "rmse_train": rmse_train, "rmse_test": rmse_test}

def augment_data(X, y, augmentation_factor, variation_pct, random_seed=42):
    """
    Augment the training data by randomly perturbing numerical features (excluding categorical ones)
    and the target by ±variation_pct.
    """
    # Set random seed for reproducibility
    np.random.seed(random_seed)
    
    n_aug = int((augmentation_factor - 1) * len(X))
    if n_aug <= 0:
        return X, y
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col]) and col not in ['cohort_month','current_month']]
    X_aug_list, y_aug_list = [], []
    for _ in range(n_aug):
        idx = np.random.randint(0, len(X))
        x_row = X.iloc[idx].copy()
        for col in numeric_cols:
            sign = np.random.choice([-1,1])
            x_row[col] = x_row[col] * (1 + sign*(variation_pct/100.0))
        X_aug_list.append(x_row)
        y_val = y.iloc[idx]
        sign = np.random.choice([-1,1])
        y_aug_list.append(y_val * (1 + sign*(variation_pct/100.0)))
    X_aug = pd.DataFrame(X_aug_list)
    y_aug = pd.Series(y_aug_list, index=X_aug.index)
    X_final = pd.concat([X, X_aug], ignore_index=True)
    y_final = pd.concat([y, y_aug], ignore_index=True)
    return X_final, y_final

def preprocess_m1(game):
    df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
    df['elapsed_months'] = df['elapsed_months'] - 1
    df = df[~((df['client_actual_spend'].fillna(0)==0) & (df['total_cac'].fillna(0)==0))]
    df['total_cac'] = df.apply(lambda row: row['client_actual_spend'] if (pd.isnull(row['total_cac']) or row['total_cac']==0) and not pd.isnull(row['client_actual_spend']) else row['total_cac'], axis=1)
    df['client_actual_spend'] = df.apply(lambda row: row['total_cac'] if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend']==0) and not pd.isnull(row['total_cac']) else row['client_actual_spend'], axis=1)
    cl1 = df.loc[:, df.columns.str.startswith('MC_')].columns.tolist()
    df = df[~df['cohort'].str.endswith(('2021','2022','2020','2019','2018'))]
    df = df[~df['cohort'].str.startswith(('2021','2022','2020','2019','2018'))]
    cols = ['game_id','total_cac','client_actual_spend','cohort','elapsed_months','transaction_period_ltv_cac','transaction_period']+cl1
    df = df[cols]
    df = df.reset_index(drop=True)
    df = df[df['game_id']==game]
    # df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    # if game == "G090":
    #     df['cohort'] = pd.to_datetime(df['cohort'], format='%d-%m-%Y')
    # else:
    #     df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    # if game == "G090":
    #     df['cohort'] = pd.to_datetime(df['cohort'], format='%d-%m-%Y')
    #     df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d-%m-%Y', errors='coerce')
    # else:
    #     df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    #     df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')
    if game == "G090":
        # Convert hyphen-based dates to slash-based dates for consistency
        df['cohort'] = df['cohort'].str.replace('-', '/')
        df['transaction_period'] = df['transaction_period'].str.replace('-', '/')
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')


    df['cohort_month'] = df['cohort'].dt.month
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)
    cat_cols = df.select_dtypes(include=["object","category"]).columns
    df[cat_cols] = df[cat_cols].fillna("missing")
    df['current_month'] = df.apply(calculate_current_month, axis=1)
    df['cohort_month'] = df['cohort_month'].astype('category')
    df['current_month'] = df['current_month'].astype('category')
    feature_columns = ['total_cac','cohort_month','current_month']+cl1
    for lag in range(1,4):
        df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
    return df, feature_columns

def forecast_cumulative_ltv_cac_variations_m1(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12):
    game_data = df[df['game_id']==game_id].copy()

    if game_id == "G090":
        game_data = game_data[game_data['cohort'] >= pd.Timestamp("2024-12-01")]

    forecasted_results = []
    sp_ltv_cac = df[df['transaction_period_ltv_cac']>0]['transaction_period_ltv_cac'].min()
    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort']==cohort].copy()
        total_cac = cohort_data['total_cac'].iloc[0]
        last_elapsed_month = int(cohort_data['elapsed_months'].max())
        cumulative_actual = 0
        actual_cumulative = []
        for month in range(0, last_elapsed_month+1):
            transactional_actual = cohort_data.loc[cohort_data['elapsed_months']==month, 'transaction_period_ltv_cac'].values
            transactional_actual = transactional_actual[0] if len(transactional_actual)>0 else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })
        lag_1 = np.nan; lag_2 = np.nan; lag_3 = np.nan; lag_1b = np.nan; lag_1g = np.nan
        cumulative_values = {'cumulative_bad': 0, 'cumulative_expected': 0, 'cumulative_good': 0}
        static_features = cohort_data.loc[cohort_data['elapsed_months']==last_elapsed_month, best_model_expected.__dict__["feature_names_in_"]].iloc[0].copy()
        for col in ['elapsed_months','current_month','ltv_lag_1','ltv_lag_2','ltv_lag_3']:
            if col in static_features:
                static_features = static_features.drop(col)
        for i in range(0, last_elapsed_month+forecast_months+1):
            elapsed_months = i
            features = static_features.copy()
            features['elapsed_months'] = elapsed_months
            if 'current_month' in best_model_expected.__dict__["feature_names_in_"]:
                cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
                current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
                features['current_month'] = current_date.month
            else:
                current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)
            features1 = features.copy(); features2 = features.copy()
            features['ltv_lag_1'] = lag_1
            features1['ltv_lag_1'] = lag_1b
            features2['ltv_lag_1'] = lag_1g
            features['ltv_lag_2'] = lag_2
            features1['ltv_lag_2'] = lag_2
            features2['ltv_lag_2'] = lag_2
            features['ltv_lag_3'] = lag_3
            features1['ltv_lag_3'] = lag_3
            features2['ltv_lag_3'] = lag_3
            features_df = pd.DataFrame([features])
            features_df1 = pd.DataFrame([features1])
            features_df2 = pd.DataFrame([features2])
            predicted_val_bad = max(0, best_model_bad.predict(features_df1)[0])
            predicted_val_expected = max(0, best_model_expected.predict(features_df)[0])
            predicted_val_good = max(0, best_model_good.predict(features_df2)[0])
            x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
            x.sort()
            predicted_val_bad, predicted_val_expected, predicted_val_good = x
            if predicted_val_good<=0: predicted_val_good = sp_ltv_cac
            if predicted_val_expected<=0: predicted_val_expected = predicted_val_good/2
            if predicted_val_bad<=0: predicted_val_bad = predicted_val_expected/2
            lag_3 = lag_2; lag_2 = lag_1; lag_1 = predicted_val_expected
            lag_1b = predicted_val_bad; lag_1g = predicted_val_good
            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m')
            }
            actual_data = next((ac for ac in actual_cumulative if ac['month']==elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac
            for model_type, predicted_val in zip(['bad','expected','good'], [predicted_val_bad, predicted_val_expected, predicted_val_good]):
                cumulative_key = f"cumulative_{model_type}"
                cumulative_values[cumulative_key] += predicted_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cumulative_values[cumulative_key]
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val
            forecasted_results.append(row_result)
        # End loop over forecast months
    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df

# --- Main Execution for M1 ---
if __name__ == "__main__":
    # Path to global config file
    config_path = "Latest Data/Segregated/config.json"
    # Load config
    try:
        with open(config_path, "r") as f:
            global_config = json.load(f)
    except Exception as e:
        print("No config found or error reading config.json.")
        global_config = {}

    # Lists to track failures and feature changes
    failed_games = []
    changed_features = {}

    # Process only configurations with keys starting with "M1_" and flag==1
    for key, cfg in global_config.items():
        if key.startswith("M1_") and cfg.get("flag", 0)==1:
            game_id = cfg["game_id"]
            selected_features = cfg["selected_features"]
            use_augmentation = cfg["use_augmentation"]
            augmentation_factor = cfg.get("augmentation_factor", 1.0)
            augmentation_pct = cfg.get("augmentation_pct", 0)
            
            print(f"Processing M1 configuration for {game_id} ...")
            
            # Set global random seeds for reproducibility
            np.random.seed(42)
            
            try:
                df, default_features = preprocess_m1(game_id)
                # Check for missing features in the CSV versus config
                missing_feats = [feat for feat in selected_features if feat not in df.columns]
                if missing_feats:
                    print(f"Warning: For game {game_id}, the following features are missing and will be ignored: {missing_feats}")
                    changed_features[game_id] = missing_feats
                    # Use only features that are present
                    selected_features = [feat for feat in selected_features if feat in df.columns]
                # Ensure 'elapsed_months' is included
                if 'elapsed_months' not in selected_features:
                    selected_features.append('elapsed_months')
                X = df[selected_features].drop(df.index[-1])
                y = df['transaction_period_ltv_cac'].drop(df.index[-1])
                from sklearn.model_selection import train_test_split
                # Time-based split (fix C2): train on earlier transaction months, test on the most recent 20%
                _tp = pd.to_datetime(df.loc[X.index, 'transaction_period'], format='%d/%m/%Y', errors='coerce')
                _order = _tp.sort_values(kind='stable').index
                _cut = int(len(_order) * 0.8)
                X_train, X_test = X.loc[_order[:_cut]], X.loc[_order[_cut:]]
                y_train, y_test = y.loc[_order[:_cut]], y.loc[_order[_cut:]]
                if use_augmentation and augmentation_factor > 1:
                    X_train, y_train = augment_data(X_train, y_train, augmentation_factor, augmentation_pct, random_seed=42)
                alphas = [0.25, 0.5, 0.75]
                models_info = {}
                for alpha in alphas:
                    print(f"Training model with alpha={alpha} for game {game_id} ...")
                    result = model_train_save_pipeline(game_id, alpha, X_train, y_train, X_test, y_test)
                    models_info[alpha] = result
                best_model_expected = models_info[0.5]["model"]
                best_model_bad = models_info[0.75]["model"]
                best_model_good = models_info[0.25]["model"]
                forecast_df = forecast_cumulative_ltv_cac_variations_m1(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12)
                output_csv = f"{game_id}_predictions.csv"
                forecast_df.to_csv(output_csv, index=False)
                print(f"Predictions saved to {output_csv}")
            except Exception as e:
                print(f"Error processing game {game_id}: {e}")
                failed_games.append(game_id)
                continue

    # After processing all games, print summary
    if failed_games:
        print("\nThe following games failed during training:")
        for game in failed_games:
            print(game)

    if changed_features:
        print("\nThe following games had missing features (these features were ignored and need to be revisited):")
        for game, features in changed_features.items():
            print(f"Game {game}: Missing features {features}")